In [ ]:
import json, re, sys, time, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

sys.path.append("../src")
from validate_adv import invalid_reason, edit_distance, ext
from evaluate import (evaluate, adversarial_metrics, deltas, adversarial_rows,
                      adversarial_recall, attack_summary, save_preds, group_mask)
from load_data import load_config

ADV_DIR, RES, MODELS = Path("../data/adversarial"), Path("../results"), Path("../models")
for p in (ADV_DIR, RES, MODELS):
    p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- config ----------------
SEED = 42                          
BUDGET_TRAIN = load_config()[0]["budget"]   
RUN_CONTROL = True                 # also train the duplicated-phishing control model
FORCE_RETRAIN = False              # reuse saved checkpoints if present
THRESHOLD, TOP_POSITIONS = 0.5, 3
NUM_EPOCHS, BATCH_SIZE, LR = 5, 256, 1e-3

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
print("Device:", device, "| train budget k =", BUDGET_TRAIN)

Device: cpu | train budget k = 3


In [5]:
with open("../data/vocab.json") as f:
    vocab = json.load(f)
with open("../data/config.json") as f:
    config = json.load(f)
index_to_string = vocab["index_to_string"]
char_to_idx = {c: i for i, c in enumerate(index_to_string)}
PAD_IDX, UNK_IDX, MAX_LEN = vocab["pad_idx"], vocab["unk_idx"], config["max_len"]
VOCAB_SIZE = len(index_to_string)

def canonicalize_model_url(url):
    url = str(url).strip()
    url = re.sub(r"^[A-Za-z][A-Za-z0-9+.\-]*://", "", url)
    url = re.sub(r"^www\d*\.", "", url, flags=re.IGNORECASE)
    return re.sub(r"/+$", "", url)

def encode_batch(urls):
    arr = np.full((len(urls), MAX_LEN), PAD_IDX, dtype=np.int64)
    for r, u in enumerate(urls):
        row = [char_to_idx.get(c, UNK_IDX) for c in u[:MAX_LEN]]
        arr[r, :len(row)] = row
    return torch.from_numpy(arr)

class URLDataset(Dataset):
    def __init__(self, df):
        self.urls, self.labels = df["URL"].tolist(), df["label"].tolist()
    def __len__(self): return len(self.urls)
    def __getitem__(self, i):
        y = 1.0 if self.labels[i] == 0 else 0.0          # phishing = 1
        return encode_batch([self.urls[i]])[0], torch.tensor(y, dtype=torch.float32)

class CharacterCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.conv1 = nn.Conv1d(embedding_dim, 128, kernel_size=5)
        self.conv2 = nn.Conv1d(128, 128, kernel_size=5)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128, 1)
    def forward(self, x):
        x = self.embedding(x).transpose(1, 2)
        x = self.relu(self.conv1(x)); x = self.relu(self.conv2(x))
        return self.fc(self.pool(x).squeeze(2))

@torch.no_grad()
def p_phish_with(m, urls, batch_size=1024):
    m.eval(); out = []
    for i in range(0, len(urls), batch_size):
        out.append(torch.sigmoid(m(encode_batch(urls[i:i + batch_size]).to(device)).squeeze(1)).cpu())
    return torch.cat(out).numpy()

# Read data

In [ ]:
# original
train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")
for d in (train_df, val_df, test_df):
    d["URL"] = d["URL"].apply(canonicalize_model_url)

# original + adversarial (generated from model 1)
train_aug = pd.read_csv(ADV_DIR / "cnn_train_original_and_adversarial.csv", keep_default_na=False)[["URL", "label"]]
n_adv_train = len(train_aug) - len(train_df)
print(f"train_original {len(train_df)} -> augmented {len(train_aug)} (+{n_adv_train} adversarial)")
print("phishing(%) in train:", round((train_df.label == 0).mean(), 4), "->", round((train_aug.label == 0).mean(), 4))
print("train_aug URLs also in test_original (check):", len(set(train_aug.URL) & set(test_df.URL)))

train_original 185397 -> augmented 198032 (+12635 adversarial)
phishing(%) in train: 0.4183 -> 0.4554
train_aug URLs also in test_original (check): 0


# Train Model 2
- validation set is same as original `val.csv`
-  training dataset flips labels to (phishing = 1) --> model outputs P(phishing). 

In [ ]:
def train_model(df_train, name, seed=SEED):
    path = MODELS / f"{name}_s{seed}.pt"
    m = CharacterCNN(VOCAB_SIZE, 32).to(device)
    if path.exists() and not FORCE_RETRAIN:
        m.load_state_dict(torch.load(path, map_location=device, weights_only=True)); print("loaded", path.name)
        return m.eval()
    set_seed(seed)
    m = CharacterCNN(VOCAB_SIZE, 32).to(device)
    crit, opt = nn.BCEWithLogitsLoss(), torch.optim.Adam(m.parameters(), lr=LR)
    tl = DataLoader(URLDataset(df_train), batch_size=BATCH_SIZE, shuffle=True)
    vl = DataLoader(URLDataset(val_df), batch_size=BATCH_SIZE, shuffle=False)
    log = []
    for ep in range(NUM_EPOCHS):
        m.train(); tr = 0
        for X, y in tl:
            X, y = X.to(device), y.to(device)
            opt.zero_grad(); loss = crit(m(X).squeeze(1), y); loss.backward(); opt.step(); tr += loss.item()
        m.eval(); va = 0
        with torch.no_grad():
            for X, y in vl:
                X, y = X.to(device), y.to(device); va += crit(m(X).squeeze(1), y).item()
        log.append({"epoch": ep + 1, "train_loss": tr / len(tl), "val_loss": va / len(vl)})
        print(f"[{name}] epoch {ep+1}/{NUM_EPOCHS} train {log[-1]['train_loss']:.4f} val {log[-1]['val_loss']:.4f}")
    torch.save(m.state_dict(), path)
    pd.DataFrame(log).to_csv(RES / f"{name}_s{seed}_train_log.csv", index=False)
    return m.eval()

model1 = CharacterCNN(VOCAB_SIZE, 32).to(device)
model1.load_state_dict(torch.load(MODELS / "cnn_baseline.pt", map_location=device, weights_only=True)); model1.eval()

model2 = train_model(train_aug, "cnn_adv_trained")

models = {"MODEL 1": model1, "MODEL 2": model2}

# train extra control model on original data + sample duplicates of phishing rows 
# so that total dataset size = size of original + adv dataset --> find out whether any gain for MODEL 2 comes from adversarial examples or just from having more phishing rows in training
if RUN_CONTROL:
    extra = train_df[train_df.label == 0].sample(n=n_adv_train, random_state=SEED)   # same count of extra phishing rows
    control_df = pd.concat([train_df, extra], ignore_index=True)
    models["CONTROL (duplicate phishing)"] = train_model(control_df, "cnn_control_dup")

# Adversarial Attack

In [ ]:
HOST_CHARS = "abcdefghijklmnopqrstuvwxyz0123456789-."
PATH_CHARS = "abcdefghijklmnopqrstuvwxyz0123456789-._~/=&?#"
START_CHARS = "/?#"

def char_mask(chars):
    mk = torch.zeros(VOCAB_SIZE, dtype=torch.bool, device=device)
    mk[[char_to_idx[c] for c in chars]] = True
    return mk
REGION_MASKS = torch.stack([char_mask(HOST_CHARS), char_mask(PATH_CHARS), char_mask(START_CHARS)])

model, E, K = model1, None, BUDGET_TRAIN

def set_attack_model(m):
    global model, E
    model = m.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    E = model.embedding.weight.detach()

def p_phish_batch(urls, batch_size=1024):
    return p_phish_with(model, urls, batch_size)

def forward_from_embeddings(emb):
    x = emb.transpose(1, 2)
    x = model.relu(model.conv1(x)); x = model.relu(model.conv2(x))
    return model.fc(model.pool(x).squeeze(2))

def logit_and_embedding_grad(url):
    ids = encode_batch([url]).to(device)
    with torch.inference_mode(False), torch.enable_grad():
        emb = model.embedding(ids).detach().clone().requires_grad_(True)
        logit = forward_from_embeddings(emb).squeeze()
        (grad,) = torch.autograd.grad(logit, emb)
    return ids[0], grad[0]

def url_layout(url):
    n = len(url)
    path_start = next((i for i, c in enumerate(url) if c in "/?#"), n)
    userinfo, at, hostport = url[:path_start].rpartition("@")
    host_start = len(userinfo) + len(at)
    host = hostport.split(":")[0]
    sub = ext(host).subdomain if host else ""
    if not sub or not host.startswith(sub):
        return n, path_start, None
    return n, path_start, (host_start, host_start + len(sub))

def ins_chars(j, layout):
    n, ps, hr = layout
    if hr and hr[0] <= j <= hr[1]: return HOST_CHARS
    if n > ps and ps + 1 <= j <= n: return PATH_CHARS
    if n == ps and j == n: return START_CHARS
    return None

def top_positions(url, layout, m=TOP_POSITIONS):
    n, ps, hr = layout
    ids, grad = logit_and_embedding_grad(url)
    pos, region = [], []
    if hr:
        for i in range(hr[0], min(hr[1], MAX_LEN)):
            pos.append(i); region.append(0)
    for i in range(ps + 1, min(n, MAX_LEN)):
        pos.append(i); region.append(1)
    if n < MAX_LEN:
        pos.append(n); region.append(2 if n == ps else 1)
    if not pos:
        return []
    pos_t = torch.tensor(pos, device=device)
    allowed = REGION_MASKS[torch.tensor(region, device=device)].clone()
    allowed[torch.arange(len(pos), device=device), ids[pos_t]] = False
    gain = -(grad @ E.T) + (grad * E[ids]).sum(1, keepdim=True)
    score = gain[pos_t].masked_fill(~allowed, float("-inf")).max(1).values
    return [pos[t] for t in score.topk(min(m, len(pos))).indices.tolist()]

def gen_candidates(url, positions, layout):
    n, ps, hr = layout
    cands = set()
    for c in PATH_CHARS:
        cands.add(url + c); cands.add(url + "/" + c)
    for i in positions:
        if i >= n:
            for c in ins_chars(n, layout): cands.add(url + c)
            continue
        chars = HOST_CHARS if (hr and hr[0] <= i < hr[1]) else PATH_CHARS
        for c in chars:
            if c != url[i]: cands.add(url[:i] + c + url[i + 1:])
        cands.add(url[:i] + url[i + 1:])
        for j in (i, i + 1):
            ic = ins_chars(j, layout)
            if ic:
                for c in ic: cands.add(url[:j] + c + url[j:])
    cands.discard(url)
    return list(cands)

def attack_url(orig, p_clean):
    url, p_cur = orig, float(p_clean)
    for _ in range(K):
        if p_cur < THRESHOLD: break
        layout = url_layout(url)
        positions = top_positions(url, layout)
        if not positions: break
        cands = gen_candidates(url, positions, layout)
        if not cands: break
        probs = p_phish_batch(cands)
        best = None
        for j in np.argsort(probs):
            if probs[j] >= p_cur: break
            if invalid_reason(orig, cands[j], K) == "":
                best = j; break
        if best is None: break
        url, p_cur = cands[best], float(probs[best])
    return {"orig_URL": orig, "URL": url, "label": 0, "p_clean": float(p_clean), "p_adv": p_cur,
            "n_edits": edit_distance(orig, url), "flipped": p_cur < THRESHOLD}

def select_targets(df):
    # all true-phishing URLs the attacked model currently detects (100%, as in the MODEL 1 test attack)
    phish = df[df["label"] == 0].copy()
    phish["p_clean"] = p_phish_batch(phish["URL"].tolist())
    det = phish[phish["p_clean"] >= THRESHOLD].reset_index(drop=True)
    print(f"phishing {len(phish)} | missed by attacked model {len(phish) - len(det)} | to attack {len(det)}")
    return det

def run_attack(targets, out_csv, chunk=2000):
    out_csv = Path(out_csv)
    done = len(pd.read_csv(out_csv, keep_default_na=False)) if out_csv.exists() else 0
    if done: print(f"{out_csv.name}: resuming {done}/{len(targets)}")
    for start in range(done, len(targets), chunk):
        part = targets.iloc[start:start + chunk]
        rows = [attack_url(u, p) for u, p in tqdm(zip(part["URL"], part["p_clean"]), total=len(part), desc=f"{out_csv.stem} {start}")]
        pd.DataFrame(rows).to_csv(out_csv, mode="a", header=not out_csv.exists(), index=False)
    return pd.read_csv(out_csv, keep_default_na=False)

In [ ]:
# load m1 adversarial examples 
adv_m1 = pd.read_csv(ADV_DIR / "cnn_test_adversarial.csv", keep_default_na=False)
adv_m1["duplicate"] = adv_m1["duplicate"].astype(str).eq("True")

def mark_duplicates(adv, split_df): 
    adv = adv.copy()
    adv["duplicate"] = (adv["n_edits"] > 0) & (adv["URL"].isin(split_df["URL"]) | adv["URL"].duplicated())
    return adv

# generate m2 adversarial examples
set_attack_model(model2)
targets_m2 = select_targets(test_df)               
adv_m2 = run_attack(targets_m2, ADV_DIR / "cnn_m2_test_adversarial.csv")

assert all(invalid_reason(o, a, K) == "" for o, a in zip(adv_m2["orig_URL"], adv_m2["URL"])), "invalid adversarial URLs"
assert (adv_m2["n_edits"] <= K).all()
adv_m2 = mark_duplicates(adv_m2, test_df)

for name, a in (("adv(M1)", adv_m1), ("adv(M2)", adv_m2)):
    s_ = attack_summary(a, K)
    print(name, {k: s_[k] for k in ("attacked", "flipped", "edited_not_flipped", "not_edited", "duplicates_dropped")},
          "| success within k:", round(s_["success_within_edits"][K], 4))
import json as _json
(RES / "cnn_m2_adv_test_summary.json").write_text(_json.dumps(attack_summary(adv_m2, K), indent=1))

# Evaluation on test set
 - Sets 2 and 3 are phishing-heavier than set 1 --> recall rises for *every* model on them (Don't read increase in recall as robustness; compare models *within* a set)
 - `fpr` on set 1 is the cost of adversarial training. 

In [ ]:
SPLIT_WEBAPP = True 
GROUPS = ["all", "web.app", "other"] if SPLIT_WEBAPP else ["all"]

def eval_set(m, df, group="all", preds_path=None):
    p = p_phish_with(m, df["URL"].tolist())
    r = evaluate(df["label"], p, urls=df["URL"], group=group)
    cm = r.pop("confusion_matrix")                         # rows = true [phish, legit]
    neg = cm[1][0] + cm[1][1]                              # legit URLs in the subset
    r["fpr"] = cm[1][0] / neg if neg else float("nan")
    r["n"] = int(sum(map(sum, cm)))
    if preds_path:
        save_preds(preds_path, df["URL"], df["label"], p)
    return r

base = test_df[["URL", "label"]]
sets = {
    "1: test_original":  base,
    "2: orig + adv(M1)": pd.concat([base, adversarial_rows(adv_m1)[["URL", "label"]]], ignore_index=True),
    "3: orig + adv(M2)": pd.concat([base, adversarial_rows(adv_m2)[["URL", "label"]]], ignore_index=True),
}
mkey = {"MODEL 1": "m1", "MODEL 2": "m2", "CONTROL (dup phishing)": "control"}

rows = []
for mn, m in models.items():
    for sn, df in sets.items():
        for g in GROUPS:
            path = RES / f"cnn_{mkey[mn]}_set{sn[0]}_preds_s{SEED}.csv" if g == "all" else None
            rows.append({"model": mn, "test set": sn, "group": g, **eval_set(m, df, g, path)})
table = pd.DataFrame(rows).set_index(["model", "test set", "group"])
display(table[["n", "accuracy", "precision", "recall", "f1", "fpr", "roc_auc"]].round(4))
table.reset_index().to_csv(RES / f"cnn_three_sets_s{SEED}.csv", index=False)

# change from set 1 to each mixed set (evaluate.deltas), per group
d_rows = []
for mn in models:
    for sn in ("2: orig + adv(M1)", "3: orig + adv(M2)"):
        for g in GROUPS:
            d_rows.append({"model": mn, "group": g, "vs set 1": sn,
                           **deltas(table.loc[(mn, "1: test_original", g)], table.loc[(mn, sn, g)])})
display(pd.DataFrame(d_rows).round(4))

# Adversarial Evaluation
- attack run on a different model --> shows transfer rate
- `adversarial_recall`: share of the edited, non-duplicate adversarial URLs the model still detects.
- `attack_success_rate` has a different denominator for each model, so it can't be compared across models

In [ ]:
def adv_eval(m, adv):
    p_clean = p_phish_with(m, adv["orig_URL"].tolist())
    p_adv = p_phish_with(m, adv["URL"].tolist())
    r = adversarial_metrics(adv["label"], p_clean, p_adv)
    r["adversarial_recall"] = adversarial_recall(p_phish_with(m, adversarial_rows(adv)["URL"].tolist()))
    return r

rows = [{"attacks from": src_, "evaluated on": mn, **adv_eval(m, adv)}
        for src_, adv in (("MODEL 1", adv_m1), ("MODEL 2", adv_m2)) for mn, m in models.items()]
adv_table = pd.DataFrame(rows)
display(adv_table.round(4))
adv_table.to_csv(RES / f"cnn_adv_only_s{SEED}.csv", index=False)

# Post-attack recall
- Every true-phishing test URL is swapped for its adversarial version if it was attacked, and then recall is taken over all true-phishing test URLs.
- fixed denominator makes MODEL 1 and MODEL 2 directly comparable.


In [ ]:
phish_urls = test_df.loc[test_df["label"] == 0, "URL"].tolist()

def post_attack_recall(m, adv, group="all"):
    final_map = dict(zip(adv["orig_URL"], adv["URL"]))
    final = [final_map.get(u, u) for u in phish_urls]
    keep = group_mask(phish_urls, group)                   # group by the original URL
    return adversarial_recall(p_phish_with(m, final)[keep])

recs = []
for mn, m in models.items():
    for g in GROUPS:
        keep = group_mask(phish_urls, g)
        recs.append({"evaluated on": mn, "group": g, "n_phishing": int(keep.sum()),
                     "clean": adversarial_recall(p_phish_with(m, phish_urls)[keep]),
                     "after adv(M1)": post_attack_recall(m, adv_m1, g),
                     "after adv(M2)": post_attack_recall(m, adv_m2, g)})
robust = pd.DataFrame(recs).set_index(["evaluated on", "group"])
display(robust.round(4))
robust.to_csv(RES / f"cnn_post_attack_recall_s{SEED}.csv")

# Compare results
- **MODEL 2 after adv(M2) vs MODEL 1 after adv(M1):** each model under a white-box attack on itself. If MODEL 2 loses less recall, adversarial training helped against an attacker who targets it.
- **MODEL 2 after adv(M1) vs MODEL 1 after adv(M1):** the easier, non-adaptive comparison. Expect it to look better, because MODEL 2 trained on perturbations from this same attack.
- **MODEL 2 vs CONTROL:** if the control matches MODEL 2, the gain is a class-prior effect, not adversarial robustness.